Реализуйте алгоритм SAC для среды lunar lander

In [ ]:
!pip install swig
!pip install "gymnasium[box2d]"

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random
from torch.distributions import Normal

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
GAMMA = 0.99
TAU = 0.005
ALPHA = 0.2
ACTOR_LR = 3e-4
CRITIC_LR = 3e-4
REPLAY_SIZE = 100000
BATCH_SIZE = 256
START_STEPS = 10000
TOTAL_STEPS = 200000
UPDATE_AFTER = 1000
UPDATE_EVERY = 50

In [ ]:
class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
        )
        self.mu_layer = nn.Linear(256, act_dim)
        self.log_std_layer = nn.Linear(256, act_dim)
        #self.act_limit = act_limit

    def forward(self, obs):
        x = F.relu(self.net(obs))
        mean, std = self.mu_layer(x),  torch.clamp(self.log_std_layer(x), -20, 2).exp()
        normal = torch.distributions.Normal(mean, std)

        x_t = normal.rsample()
        y_t = torch.tanh(x_t)
        action = y_t * (action_high - action_low) / 2.0 + (action_low + action_high) / 2.0

        log_prob = normal.log_prob(x_t)
        log_prob -= torch.log((1 - y_t.pow(2)) + 1e-6)
        log_prob = log_prob.sum(1, keepdim=True)

        return action, log_prob

    def get_action(self, obs, deterministic=False):
        # Напишите функцию, которая возвращает только действие. Если deterministic True, то действие не семплируется, а просто берётся mean (его всё ещё надо предобразовать)
        x = self.net(obs)
        mean = self.mu_layer(x)
        log_std = torch.clamp(self.log_std_layer(x), -20, 2)
        std = log_std.exp()

        if deterministic:
            x_t = mean
        else:
            normal = Normal(mean, std)
            x_t = normal.rsample()

        y_t = torch.tanh(x_t)

        action = y_t * (action_high - action_low) / 2.0 + (action_high + action_low) / 2.0
        return action

In [ ]:
class Critic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.q1 = nn.Sequential(
            nn.Linear(obs_dim + act_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )
        self.q2 = nn.Sequential(
            nn.Linear(obs_dim + act_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, obs, act):
        x = torch.cat([obs, act], dim=-1)
        return self.q1(x), self.q2(x)

In [ ]:
class ReplayBuffer:
    def __init__(self, size):
        self.buffer = deque(maxlen=size)

    def add(self, *args):
        self.buffer.append(tuple(args))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = map(np.array, zip(*batch))
        return (
            torch.tensor(states, dtype=torch.float32),
            torch.tensor(actions, dtype=torch.float32),
            torch.tensor(rewards, dtype=torch.float32).unsqueeze(1),
            torch.tensor(next_states, dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32).unsqueeze(1)
        )
    def __len__(self):
      return len(self.buffer)

In [ ]:
env = gym.make("LunarLanderContinuous-v3")
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]
action_low, action_high = float(env.action_space.low[0]), float(env.action_space.high[0])

actor = Actor(obs_dim, act_dim)
critic = Critic(obs_dim, act_dim)
critic_target = Critic(obs_dim, act_dim)
critic_target.load_state_dict(critic.state_dict())

actor_opt = optim.Adam(actor.parameters(), lr=ACTOR_LR)
critic_opt = optim.Adam(critic.parameters(), lr=CRITIC_LR)

replay = ReplayBuffer(REPLAY_SIZE)

obs, _ = env.reset()
episode_return, episode_len = 0, 0

In [ ]:
def update():
  if len(replay) < BATCH_SIZE:
        return

  state, action, reward, next_state, done = replay.sample(BATCH_SIZE)
  with torch.no_grad():
      next_action, next_log_prob = actor.forward(next_state)
      q1_target, q2_target = critic_target(next_state, next_action)
      q_target = torch.min(q1_target, q2_target) - ALPHA * next_log_prob
      target = reward + GAMMA * q_target * (1 - done)

  q1, q2 = critic(state, action)
  critic_loss = F.mse_loss(q1, target) + F.mse_loss(q2, target)
  critic_opt.zero_grad()
  critic_loss.backward()
  critic_opt.step()

  new_action, log_prob = actor.forward(state)
  q1_pi, q2_pi = critic(state, new_action)
  actor_loss = (ALPHA * log_prob - torch.min(q1_pi, q2_pi)).mean()
  actor_opt.zero_grad()
  actor_loss.backward()
  actor_opt.step()

  for param, target_param in zip(critic.parameters(), critic_target.parameters()):
      target_param.data.copy_(TAU * param.data + (1 - TAU) * target_param.data)

In [ ]:
for step in range(TOTAL_STEPS):
    if step < START_STEPS:
        act = env.action_space.sample()
    else:
        with torch.no_grad():
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            act = actor.get_action(obs_t).cpu().numpy()[0]

    next_obs, rew, terminated, truncated, _ = env.step(act)
    done = terminated or truncated
    replay.add(obs, act, rew, next_obs, done)

    obs = next_obs
    episode_return += rew
    episode_len += 1

    if done:
        obs, _ = env.reset()
        print(f"Step: {step}, Return: {episode_return:.2f}, Len: {episode_len}")
        episode_return, episode_len = 0, 0

    if step >= UPDATE_AFTER and step % UPDATE_EVERY == 0:
        for _ in range(UPDATE_EVERY):
            update()

Step: 37, Return: -159.66, Len: 118
Step: 117, Return: -199.31, Len: 80
Step: 210, Return: -257.95, Len: 93
Step: 353, Return: -419.49, Len: 143
Step: 449, Return: -122.50, Len: 96
Step: 542, Return: -310.56, Len: 93
Step: 611, Return: -119.15, Len: 69
Step: 755, Return: -27.44, Len: 144
Step: 861, Return: -177.94, Len: 106
Step: 977, Return: -331.54, Len: 116
Step: 1072, Return: -280.06, Len: 95
Step: 1157, Return: -326.04, Len: 85
Step: 1260, Return: -63.56, Len: 103
Step: 1362, Return: -108.92, Len: 102
Step: 1433, Return: -25.82, Len: 71
Step: 1538, Return: -422.60, Len: 105
Step: 1642, Return: -418.61, Len: 104
Step: 1736, Return: -358.84, Len: 94
Step: 1813, Return: -75.29, Len: 77
Step: 1967, Return: -375.37, Len: 154
Step: 2119, Return: -142.49, Len: 152
Step: 2207, Return: -255.20, Len: 88
Step: 2286, Return: -51.45, Len: 79
Step: 2406, Return: -89.24, Len: 120
Step: 2494, Return: -83.11, Len: 88
Step: 2589, Return: -53.51, Len: 95
Step: 2706, Return: -185.43, Len: 117
Step: 2